In [1]:
import os
import re
import time
import spacy
import regex
import pickle
import string
import sqlite3
import inflect
import typing
import ahocorasick
import numpy as np
import pandas as pd
from tqdm import tqdm
from os import listdir
from itertools import islice, chain
from pandas import DataFrame
from taxonerd import TaxoNERD
from functools import lru_cache
from rich.console import Console
from os.path import isfile, join
from spacy.util import filter_spans
from IPython.display import clear_output
from spacy.tokens import Token, Doc, Span
from typing import Any, List, Dict, Tuple, Set, Optional, Union, Literal, Callable, TypedDict, Sequence
import unicodedata
from tqdm.auto import tqdm

c:\Users\lbeln\anaconda3\envs\3.10_CUDA\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
inflector = inflect.engine()


@lru_cache(maxsize=64)
def plural(s: Any) -> str | None:
    try:
        return inflector.plural(s) or None
    except Exception:
        return None


@lru_cache(maxsize=64)
def singular(s: Any) -> str | None:
    try:
        return inflector.singular_noun(s) or None
    except Exception:
        return None


@lru_cache(maxsize=64)
def get_inflections(s: str, tag: str | None) -> Set[str]:
    infs = set()

    if (not tag or tag == 'NN' or tag == 'NNP') and (inf := plural(s)) and inf != s:
        infs.add(inf)
    
    if (not tag or tag == 'NNS' or tag == 'NNPS') and (inf := singular(s)) and inf != s:
        infs.add(inf)
    
    return infs

In [3]:
latin_inflections = [('pus', 'podes'), ('ex', 'ices'), ('ix', 'ices'), ('us', 'i'), ('um', 'a'), ('a', 'ae'), ('is', 'es'), ('x', 'ges'), ('on', 'a'), ('os', 'oi'), ('ma', 'mata')]


@lru_cache(maxsize=64)
def get_inflections_latin(s: str) -> Set[str]:
    inflections = set()

    for k, v in latin_inflections:
        inf = re.sub(rf'{k}$', v, s)
        if inf != s:
            inflections.add(inf)
        
        inf = re.sub(rf'{v}$', k, s)
        if inf != s:
            inflections.add(inf)
        
    return inflections

In [4]:
sp = 'sp.'
spp = 'spp.'
binomial = r'((^\p{Lu}(\.|\p{Ll}+))\s(sp\.|spp\.|(\p{Ll}(\.|\p{Ll}+))))'
trinomial = r'((^\p{Lu}(\.|\p{Ll}+))\s(\p{Ll}(\.|\p{Ll}+)\s)(sp\.|spp\.|(\p{Ll}(\.|\p{Ll}+))))'


@lru_cache(maxsize=128)
def re_taxa(s: str) -> bool:
    return bool(regex.match(r'^\p{Lu}\p{Ll}+$', s))


@lru_cache(maxsize=128)
def re_taxonomic_name(s: str) -> bool:
    return bool(regex.match(r'^(\p{Lu}(\.|\p{Ll}+))\s(\p{Ll}(\.|\p{Ll}+)\s)?\p{Ll}(\.|\p{Ll}+)$', s))


@lru_cache(maxsize=128)
def re_taxonomic_notation(s: str) -> str | None:
    if match := regex.match(rf'{binomial}\s((\p{{Lu}}|\(|et al|sensu|nov|comb|stat|syn|s\.s\.|s\.l\.).*)$', s):
        return match.group(1)

    if match := regex.match(rf'{trinomial}\s((\p{{Lu}}|\(|et al|sensu|nov|comb|stat|syn|s\.s\.|s\.l\.).*)$', s):
        return match.group(1)

    return None


@lru_cache(maxsize=128)
def abbreviate(s: str) -> List[str]:
    name = s if re_taxonomic_name(s) else re_taxonomic_notation(s)
    if not name:
        return []
    
    res = []
    name = name.split()
    
    if len(name) >= 2:
        res.append(f'{name[0][0].upper()}. {name[1].lower()}')
    
    if len(name) == 3:
        res.append(f'{name[0][0].upper()}. {name[1][0].lower()}. {name[2].lower()}')
    
    return res

In [5]:
conn = sqlite3.connect('./db/COL.db')
# conn.execute("ANALYZE NameUsage")
# conn.execute("PRAGMA cache_size = -1000000")  # Moooore Memory


def load_data(path: str, load: Callable[[], Any], *, force: bool = False) -> Any:
    data = None
    if not force and os.path.exists(path):
        with open(path, "rb") as file:
            data = pickle.load(file)
    else:
        data = load()
        with open(path, "wb") as file:
            pickle.dump(data, file, pickle.HIGHEST_PROTOCOL)
    return data

In [6]:
def load_scientific_groups():
    df = pd.read_sql(f"""
        SELECT *
        FROM NameUsage
    """, conn)

    cols = [
        ["kingdom", "col:kingdom"],
        ["phylum", "col:phylum"],
        ["subphylum", "col:subphylum"],
        ["class", "col:class"],
        ["subclass", "col:subclass"],
        ["order", "col:order"],
        ["suborder", "col:suborder"],
        ["superfamily", "col:superfamily"],
        ["family", "col:family"],
        ["subfamily", "col:subfamily"],
        ["genus", "col:genus"],
        ["subgenus", "col:subgenus"],
        ["genericName", "col:genericName"],
        ["specificEpithet", "col:specificEpithet"],
        ["intraspecificEpithet", "col:infraspecificEpithet"],
        ["species", "col:species"],
        ["scientificName", "col:scientificName"]
    ]
    
    data = []
    for col in cols:
        vals = set()
        
        for val in df[col[1]].tolist():
            if not val:
                continue
            val = val.lower()
            # The sub-genus is placed inside '(...)',
            # so we need to capture it.
            if col[0] == "subgenus":
                match = regex.match(r'^([A-Za-z]+)\s\(([A-Za-z]+)\)$', val)
                vals.add(val if not match else match.group(2))
            else:
                vals.add(val)
        
        data.append({
            'label': col[0],
            'column': col[1],
            'values': vals
        })

    del df
    return data

scientific_groups = load_data("./db/ScientificGroups.pickle", load_scientific_groups, force=False)
print(f'# of Groups: {len(scientific_groups)}')

# of Groups: 17


In [7]:
def load_vernacular_names():
    df = pd.read_sql(f"""
        SELECT "col:name" 
        FROM VernacularName 
        WHERE "col:language" = 'eng'
    """, conn)

    names = [{name} | get_inflections(name, None) for name in df["col:name"].tolist() if name]
    names = set().union(*names)
    names = names | {
        # Mammals
        "mammal", "mammals",
        "rodent", "rodents",
        "primate", "primates",
        "marsupial", "marsupials",
        "bat", "bats",
        "whale", "whales",
        "dolphin", "dolphins",
        "porpoise", "porpoises",
        "seal", "seals",
        "otter", "otters",
        "bear", "bears",
        "cat", "cats",
        "dog", "dogs",
        "wolf", "wolves",
        "fox", "foxes",
        "deer", "deers",
        "antelope", "antelopes",
        "monkey", "monkeys",
        "ape", "apes",
        "shrew", "shrews",
        "mole", "moles",
        "hedgehog", "hedgehogs",
        "rabbit", "rabbits",
        "hare", "hares",
        "squirrel", "squirrels",
        "rat", "rats",
        "mouse", "mice",
        "vole", "voles",
        "beaver", "beavers",
        "pig", "pigs",
        "horse", "horses",
        "cow", "cows",
        "sheep", "sheeps",
        "goat", "goats",
        # Birds
        "bird", "birds",
        "seabird", "seabirds",
        "shorebird", "shorebirds",
        "songbird", "songbirds",
        "raptor", "raptors",
        "waterfowl",
        "wader", "waders",
        "hawk", "hawks",
        "eagle", "eagles",
        "owl", "owls",
        "parrot", "parrots",
        "pigeon", "pigeons",
        "dove", "doves",
        "duck", "ducks",
        "goose", "geese",
        "swan", "swans",
        "heron", "herons",
        "gull", "gulls",
        "tern", "terns",
        "kingfisher", "kingfishers",
        "woodpecker", "woodpeckers",
        "finch", "finches",
        "sparrow", "sparrows",
        "warbler", "warblers",
        "thrush", "thrushes",
        "robin", "robins",
        "starling", "starlings",
        "swift", "swifts",
        "swallow", "swallows",
        "martin", "martins",
        "crane", "cranes",
        "stork", "storks",
        "flamingo", "flamingos",
        "penguin", "penguins",
        "albatross", "albatrosses",
        "petrel", "petrels",
        "cormorant", "cormorants",
        "pelican", "pelicans",
        "sunbird", "sunbirds",
        "oystercatcher", "oystercatchers",
        # Reptiles
        "reptile", "reptiles",
        "snake", "snakes",
        "lizard", "lizards",
        "gecko", "geckos",
        "skink", "skinks",
        "turtle", "turtles",
        "tortoise", "tortoises",
        "crocodile", "crocodiles",
        "alligator", "alligators",
        "chameleon", "chameleons",
        # Amphibians
        "amphibian", "amphibians",
        "frog", "frogs",
        "toad", "toads",
        "salamander", "salamanders",
        "newt", "newts",
        "caecilian", "caecilians",
        # Fish
        "fish", "fishes",
        "shark", "sharks",
        "ray", "rays",
        "eel", "eels",
        "salmon", "salmons",
        "trout", "trouts",
        "cod", "cods",
        "tuna", "tunas",
        "herring", "herrings",
        "anchovy", "anchovies",
        "carp", "carps",
        "catfish", "catfishes",
        "perch", "perches",
        "bass", "basses",
        "flatfish", "flatfishes",
        "pufferfish", "pufferfishes",
        "goby", "gobies",
        # Insects
        "insect", "insects",
        "bug", "bugs",
        "beetle", "beetles",
        "fly", "flies",
        "moth", "moths",
        "butterfly", "butterflies",
        "bee", "bees",
        "wasp", "wasps",
        "ant", "ants",
        "termite", "termites",
        "dragonfly", "dragonflies",
        "damselfly", "damselflies",
        "grasshopper", "grasshoppers",
        "cricket", "crickets",
        "locust", "locusts",
        "louse", "lice",
        "flea", "fleas",
        "tick", "ticks",
        "mite", "mites",
        "aphid", "aphids",
        "cicada", "cicadas",
        "mantis", "mantises",
        "cockroach", "cockroaches",
        "earwig", "earwigs",
        "mayfly", "mayflies",
        "stonefly", "stoneflies",
        "lacewing", "lacewings",
        "weevil", "weevils",
        "moth", "moths",
        # Other Invertebrates
        "invertebrate", "invertebrates",
        "macroinvertebrate", "macroinvertebrates", 
        "microinvertebrate", "microinvertebrates", 
        "spider", "spiders",
        "scorpion", "scorpions",
        "crab", "crabs",
        "lobster", "lobsters",
        "shrimp", "shrimps",
        "prawn", "prawns",
        "barnacle", "barnacles",
        "woodlouse", "woodlice",
        "millipede", "millipedes",
        "centipede", "centipedes",
        "worm", "worms",
        "earthworm", "earthworms",
        "leech", "leeches",
        "snail", "snails",
        "slug", "slugs",
        "clam", "clams",
        "mussel", "mussels",
        "oyster", "oysters",
        "scallop", "scallops",
        "squid", "squids",
        "octopus", "octopuses",
        "cuttlefish", "cuttlefishes",
        "nautilus", "nautiluses",
        "jellyfish", "jellyfishes",
        "coral", "corals",
        "anemone", "anemones",
        "sponge", "sponges",
        "starfish", "starfishes",
        "urchin", "urchins",
        "sealion", "sealions",
        "sea lion", "sea lions",
        # Plants
        "plant", "plants",
        "tree", "trees",
        "shrub", "shrubs",
        "herb", "herbs",
        "grass", "grasses",
        "flower", "flowers",
        "weed", "weeds",
        "vine", "vines",
        "fern", "ferns",
        "moss", "mosses",
        "liverwort", "liverworts",
        "hornwort", "hornworts",
        "conifer", "conifers",
        "palm", "palms",
        "succulent", "succulents",
        "cactus", "cacti", "cactuses",
        "orchid", "orchids",
        "sedge", "sedges",
        "rush", "rushes",
        "reed", "reeds",
        "lichen", "lichens",
        "seagrass", "seagrasses",
        "seaweed", "seaweeds",
        # Fungi
        "fungus", "fungi",
        "mushroom", "mushrooms",
        "mould", "moulds",
        "mold", "molds",
        "yeast", "yeasts",
        # Microorganisms
        "bacterium", "bacteria",
        "microbe", "microbes",
        "microorganism", "microorganisms",
        "protist", "protists",
        "alga", "algae",
        "diatom", "diatoms",
        "amoeba", "amoebas", "amoebae",
        "virus", "viruses",
        "parasite", "parasites",
        "zooplankton"
    }

    del df
    return names

vernacular_names = load_data("./db/VernacularNames.pickle", load_vernacular_names, force=False)
print(f'# of Names: {len(vernacular_names)}')

# of Names: 377699


In [8]:
def load_mapped_names():
    df = pd.read_sql(f'''
        SELECT *
        FROM MappedName
    ''', conn)

    m = {}
    
    for i, row in df.iterrows():
        names = [name for name in [row.ScientificName, row.VernacularName, row.Genus] if name]

        for i, name in enumerate(names):
            forms = [name.lower()]
            
            # Add Abbreviated Scientific Name
            if i == 0 and (abbrevs := abbreviate(name)):
                forms.append(abbrevs[-1].lower())

            for form in forms:
                if form not in m:
                    m[form] = set()
                
                for col in [col for col in row if col]:
                    m[form].add(col.lower())
    
    del df
    return m

mapped_names = load_data("./db/MappedNames.pickle", load_mapped_names, force=False)
print(f'# of Names: {len(mapped_names)}')

# of Names: 404008


In [9]:
roles = {
    'carnivores',
    'carnivore',
    'predators',
    'predator',
    'species',
    'prey',
    'grazer',
    'grazers',
    'producer',
    'producers',
    'consumer',
    'consumers',
    'competitor',
    'competitors',
}

In [10]:
def load_all_names():
    all_names = ahocorasick.Automaton()
     
    # Scientific Names
    for group in scientific_groups:
        for val in group['values']:
            all_names.add_word(val, {
                'key': val,
                'label': 's',
            })
    
     # Role
    for role in roles:
        all_names.add_word(role, {
            'key': role,
            'label': 'r'
        })
    
    # Vernacular Names
    for name in vernacular_names:
        all_names.add_word(name, {
            'key': name,
            'label': 'v'
        })
    
    all_names.make_automaton()
    return all_names

all_names = load_data("./db/AllNames.pickle", load_all_names, force=False)
print(f'# of Names: {len(all_names)}')

# of Names: 8836299


In [11]:
@lru_cache(maxsize=64)
def in_roles(s: str) -> bool:
    return s.lower() in roles


@lru_cache(maxsize=64)
def in_vernacular(s: str) -> bool:
    return s.lower() in vernacular_names


@lru_cache(maxsize=64)
def in_scientific(s: str) -> str | None:
    s = s.lower()
    for group in scientific_groups:
        if s in group['values']:
            return group['label']
    return None

In [12]:
def get_substitutions(s: str) -> list[str]:
    return mapped_names.get(s.lower(), [])

In [13]:
@lru_cache(maxsize=64)
def is_taxa(s: str) -> bool:
    return re_taxa(s) and (data := all_names.get(s, None)) and data['label'] == 's'


@lru_cache(maxsize=64)
def is_taxonomic(s: str) -> bool:
    return re_taxonomic_name(s) and (data := all_names.get(s, None)) and data['label'] == 's'


@lru_cache(maxsize=64)
def is_taxonomic_notation(s: str) -> str | None:
    name = re_taxonomic_notation(s)
    return name if name and all_names.get(name, None) else None

In [14]:
bacteria_kingdoms = {'pseudomonadati', 'bacillati', 'methanobacteriati', 'fusobacteriati', 'thermotogati'}


@lru_cache(maxsize=128)
def names_related(s1: str, s2: str) -> bool:
    buffer = []

    s1 = s1.lower()
    s2 = s2.lower()
    
    if s1 == s2:
        return True
    
    used_s1 = False
    used_s2 = False
    
    for group in scientific_groups:
        if used_s1 and used_s2:
            break
        
        if not used_s1 and s1 in group['values']:
            buffer.append((s1, group['column']))
            used_s1 = True
            continue

        if not used_s2 and s2 in group['values']:
            buffer.append((s2, group['column']))
            used_s2 = True
            continue
        
    if len(buffer) != 2:
        return False
        
    (t1_val, t1_col), (t2_val, t2_col) = buffer
    
    # Bacteria
    # We need a different method of determining whether
    # something is related to bacteria, as bacteria is
    # not really defined in the database.
    if t1_val == 'bacteria' and t2_col in ['col:scientificName', 'col:species']:
        q = f'''
            SELECT LOWER("col:kingdom")
            FROM NameUsage
            WHERE LOWER("{t2_col}") = \'{t2_val}\'
            LIMIT 1
        '''
        output = conn.execute(q).fetchone()
        output = None if not output or not output[0] else output[0].lower()
        if output in bacteria_kingdoms:
            return True
    

    q = f'''
        SELECT 1
        FROM NameUsage
        WHERE LOWER("{t2_col}") = \'{t2_val}\' AND LOWER("{t1_col}") = \'{t1_val}\'
        LIMIT 1
    '''
    return bool(conn.execute(q).fetchone())

In [15]:
def expand_unit(doc: Doc, il_unit: int, ir_unit: int, il_boundary: int, ir_boundary: int, speech: List[str] = [], literals: List[str] = [], include: bool = True, direction: str = 'BOTH', verbose: bool = False):    
    if il_unit > ir_unit:
        print(f"Error: il_unit of {il_unit} greater than ir_unit of {ir_unit}")
        return None
    
    if direction in ['BOTH', 'LEFT'] and il_boundary > il_unit:
        print(f"Error: il_unit of {il_unit} less than il_boundary of {il_boundary}")
        return None
    
    if direction in ['BOTH', 'RIGHT'] and ir_boundary < ir_unit:
        print(f"Error: ir_unit of {ir_unit} greater than ir_boundary of {ir_boundary}")
        return None
    
    # Move Left
    if direction in ['BOTH', 'LEFT']:
        # The indices are inclusive, therefore, when 
        # the condition fails, il_unit will be equal
        # to il_boundary.
        while il_unit > il_boundary:
            # We assume that the current token is allowed,
            # and look to the token to the left.
            l_token = doc[il_unit-1]

            # If the token is invalid, we stop expanding.
            in_set = l_token.pos_ in speech or l_token.lower_ in literals

            # Case 1: include=False, in_set=True
            # If we're not meant to include the defined tokens, and the
            # current token is in that set, we stop expanding.
            # Case 2: include=True, in_set=False
            # If we're meant to include the defined tokens, and the current
            # token is not in that set, we stop expanding.
            # Case 3: include=in_set
            # If we're meant to include the defined tokens, and the current
            # token is in that set, we continue expanding. If we're not meant
            # to include the defined tokens, and the current token is not
            # in that set, we continue expanding.
            if include ^ in_set:
                break
            
            # Else, the left token is valid, and
            # we continue to expand.
            il_unit -= 1

    # Move Right
    if direction in ['BOTH', 'RIGHT']:
        # Likewise, when the condition fails,
        # ir_unit will be equal to the ir_boundary.
        # The ir_boundary is also inclusive.
        while ir_unit < ir_boundary:
            # Assuming that the current token is valid,
            # we look to the right to see if we can
            # expand.
            r_token = doc[ir_unit+1]

            # If the token is invalid, we stop expanding.
            in_set = r_token.pos_ in speech or r_token.lower_ in literals
            if include ^ in_set:
                break

            # Else, the token is valid and
            # we continue.
            ir_unit += 1

    assert il_unit >= il_boundary and ir_unit <= ir_boundary
    expanded_unit = doc[il_unit:ir_unit+1]
        
    return expanded_unit

In [16]:
def contract_unit(doc: Doc, il_unit: int, ir_unit: int, speech: List[str] = [], literals: List[str] = [], include: bool = True, direction: str = 'BOTH', verbose: bool = False): 
    if il_unit > ir_unit:
        print(f"Error: il_unit of {il_unit} greater than ir_unit of {ir_unit}")
        return None
    
    # Move Right
    if direction in ['BOTH', 'LEFT']:
        while il_unit < ir_unit:
            # We must check if the current token is not allowed. If it's
            # not allowed, we contract (remove).
            token = doc[il_unit]

            # include = True means that we want the tokens that match
            # the speech and/or literals in the contracted unit.
            
            # include = False means that we don't want the tokens that
            # match the speech and/or literals in the contracted unit.
            
            # Case 1: include = True, in_set = True
            # We have a token that's meant to be included in the set.
            # However, we're contracting, which means we would end up
            # removing the token if we continue. Therefore, we break.
            
            # Case 2: include = False, in_set = False
            # We have a token that's not in the set which defines the
            # tokens that aren't meant to be included. Therefore, we 
            # have a token that is meant to be included. If we continue,
            # we would end up removing this token. Therefore, we break.
            
            # Default:
            # If we have a token that's in the set (in_set=True) of
            # tokens we're not supposed to include in the contracted 
            # unit (include=False), we need to remove it. Likewise, if
            # we have a token that's not in the set (in_set=False) of
            # tokens to include in the contracted unit (include=True),
            # we need to remove it.
            
            in_set = token.pos_ in speech or token.lower_ in literals
            if include == in_set:
                break

            # The token is valid, thus we continue.
            il_unit += 1

    # Move Left      
    if direction in ['BOTH', 'RIGHT']:
        while ir_unit > il_unit:
            token = doc[ir_unit]

            # The token is invalid and we
            # stop contracting.
            in_set = token.pos_ in speech or token.lower_ in literals
            if include == in_set:
                break

            # The token is valid and we continue.
            ir_unit -= 1

    assert il_unit <= ir_unit
    contracted_unit = doc[il_unit:ir_unit+1]
    
    return contracted_unit

In [17]:
def break_text(text: str) -> List[str]:
    enclosures = {
        "(":")", 
        "[":"]",
        "{":"}"
    }
    
    # This contains the text that's not inside
    # any enclosure.
    base = []

    # This contains the text that's inside
    # an enclosure.
    groups = []
    
    # This is used for building groups, which can
    # have a nested structure.
    stacks = []
    
    # These are the pairs of characters that
    # define the enclosure (parenthetical).
    openers = list(enclosures.keys())
    closers = list(enclosures.values())
    
    # This contains the opening characters of the groups 
    # that are currently open (e.g. '(', '['). We use it 
    # so that we know whether to open or close a group.
    opened = []
    
    for i, char in enumerate(text):
        # Open Group
        if char in openers:
            stacks.append([])
            opened.append(char)
        # Close Group
        elif opened and char == enclosures.get(opened[-1], ""):
            groups.append(stacks.pop())
            opened.pop()
        # Add to Group
        elif opened:
            stacks[-1].append(i)
        # Add to Base Text
        else:
            base.append(i)
    
    # We close the remaining groups that have not
    # been closed.
    while stacks:
        groups.append(stacks.pop())
        
    # Cluster Groups' Indices
    # A list in the lists of indices (where each list represents a group of text) could have 
    # an interruption (e.g. [0, 1, 2, 10, 15]) because of a parenthetical. So, we cluster the
    # indices in each list to make the output more useful (e.g. [(0, 3), (10, 16)]). As you
    # can see, we've adjusted some indices for ease-of-use.
    lists_of_indices = [*groups, base]        
    lists_of_clustered_indices = []

    for list_of_indices in lists_of_indices:
        if not list_of_indices:
            continue

        # We start off with a single cluster that is made up of the
        # first index. If the next index follows the first index, 
        # we continue the cluster. If it doesn't, we create a new cluster.
        clustered_indices = [[list_of_indices[0], list_of_indices[0] + 1]]
        
        for index in list_of_indices[1:]:
            if clustered_indices[-1][1] == index:
                clustered_indices[-1][1] = index + 1
            else:
                clustered_indices.append([index, index + 1])

        # Add Clustered Indices
        lists_of_clustered_indices.append(clustered_indices)

    lists_of_clustered_text = [[text[cluster[0]:cluster[1]] for cluster in clusters] for clusters in lists_of_clustered_indices]
    return ["".join(list_of_clustered_text) for list_of_clustered_text in lists_of_clustered_text]

In [18]:
def clean_text(text: str) -> str:
    # 1. Delete Inside and Outside Space
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([?.!,])", r"\1", text)
    text = text.strip()

    # 2. Delete Outside Non-Letters
    while text:
        start_len = len(text)
        if text and not text[0].isalnum():
            text = text[1:]
        if text and not text[-1].isalnum():
            text = text[:-1]
        if start_len == len(text):
            break
    
    return text

In [19]:
def is_boundary(c):
    return c.isspace() or c in string.punctuation

In [20]:
class Entity:
    def __init__(self, doc: Doc) -> None:
        self.doc = doc
        self.doc_text = doc.text
        self.doc_text_lower = doc.text.lower()
    

    def search_strings_tn(self) -> Set[str]:
        search_strings = set()
        
        for ent in self.doc.ents:
            # We break the text into texts if it has a parenthetical.
            # So, 'cat (dog)' becomes ['cat', 'dog'].
            texts = break_text(self.doc_text_lower[ent.start_char:ent.end_char])
            texts = list(clean_text(text) for text in texts)
            search_strings.update(texts)

            for text in texts:
                if not text or re.search(r"[^\w\s.-]", text):
                    continue
                
                # Add Base of Text (Ending Noun)
                b_text = text.split()[-1]
                if not re.match(r"\w\$.", b_text):
                    search_strings.add(b_text)

                # Add Singular Version
                s_text = singular(text)
                if s_text:
                    search_strings.add(s_text)

                # Add Plural Version
                p_text = plural(text)
                if p_text:
                    search_strings.add(p_text)
        
        return search_strings


    def find_matches_tn(self, search_strings: Set[str]) -> List[Tuple[int, int]]:
        all_matches = []
        for search in search_strings:
            matches = re.finditer(re.escape(search), self.doc_text_lower, re.IGNORECASE)
            all_matches.extend((match.start(), match.end()) for match in matches)
        return all_matches


    def find_matches_db(self) -> List[Tuple[int, int]]:
        matches = []
        for r_i, data in all_names.iter(self.doc_text_lower):
            key = data['key']
            matches.append((r_i - len(key) + 1, r_i + 1))
        return matches


    def find_ents_from_matches(self, matches: List[Tuple[int, int]]):
        spans = []
        
        for l_index, r_index in matches:
            # The full word must match, not just a substring inside of it.
            # So, if the species we're looking for is "ant", only "ant"
            # will match -- not "pants" or "antebellum". Therefore, the
            # characters to the left and right of the matched string cannot
            # be letters.
            match = self.doc_text[l_index:r_index]
            match_lower = self.doc_text_lower[l_index:r_index]

            l_bound = l_index == 0 or is_boundary(self.doc_text_lower[l_index-1])
            r_bound = r_index == len(self.doc_text_lower) or is_boundary(self.doc_text_lower[r_index])
            
            if not l_bound or not r_bound or r_index - l_index <= 1:
                continue
            
            # Check 1: No Bad Characters
            match = self.doc_text[l_index:r_index]
            if re.search(r"[^\w\s.-]", match):
                continue
            
            # Check 2: Correct Casing, if Scientific
            if data := all_names.get(match_lower, None):
                if data['label'] == 's' and not match[0].isupper():
                    continue
            
            span = self.doc.char_span(l_index, r_index, alignment_mode="expand")
            if not span or not len(span):
                continue
            
            not_sent_start = span.sent.start != span.start

            # Check 3: Correct Casing and Types, Vernacular and Role
            if in_vernacular(match_lower) or in_roles(match_lower):
                if not any(token.pos_ == 'NOUN' for token in span):
                    continue

                if not_sent_start and span[0].pos_ == 'NOUN' and match[0].isupper():
                    continue
            
            # Check 4: Correct Casing, Scientific
            if data := all_names.get(match_lower, None):
                if not_sent_start and in_scientific(match_lower) in ['specificEpithet', 'intraspecificEpithet'] and match[0].isupper():
                    continue
            
            # Expand Species
            # Let's say there's a word like "squirrel". That's a bit ambiguous. 
            # Is it a brown squirrel, a bonobo? If the species is possibly missing
            # information (like an adjective to the left of it), we should expand
            # in order to get a full picture of the species.
            is_short = len(span) == 1 and span[0].pos_ == "NOUN"
            
            # Remove Outer Symbols
            # There are times where a species is identified with a parenthesis
            # nearby. Here, we remove that parenthesis (and any other symbols).

            span = contract_unit(
                self.doc,
                il_unit=span.start, 
                ir_unit=span.end-1, 
                speech=["ADJ", "PROPN", "NOUN", "X"],
                include=True,
                verbose=False
            )

            if not span or not len(span):
                continue
            
            if is_short:
                span = expand_unit(
                    self.doc,
                    il_unit=span.start, 
                    ir_unit=span.end-1,
                    il_boundary=0,
                    ir_boundary=len(self.doc),
                    speech=["ADJ", "PROPN", "NOUN"],
                    literals=["-"],
                    include=True,
                    direction="LEFT",
                    verbose=False
                )

                if not span:
                    continue
                
                # Remove Outer Symbols (Again)
                # The previous expansion contains a literal.
                # There might not be a need for that literal.
                # To handle that case, we contract.
                span = contract_unit(
                    self.doc,
                    il_unit=span.start,
                    ir_unit=span.end-1,
                    speech=["ADJ", "PROPN", "NOUN", "X"],
                    include=True,
                    verbose=False
                )

                if not span or not len(span):
                    continue
            
            # A species must have a noun or a
            # proper noun. This may help discard
            # bad results.
            noun_found = False
            for token in span:
                if token.pos_ in ["NOUN", "PROPN", "X"]:
                    noun_found = True
                    break
            
            if not noun_found:
                continue

            # The names I'd like to identify should
            # not have any numbers or odd characters.
            if re.search(r"[^\w\s.-]", match):
                continue
            
            # Added
            spans.append(span)

        spans = filter_spans(spans)
        return spans


    def find_ents_tn(self) -> List[Span]:
        searches = self.search_strings_tn()
        matches = self.find_matches_tn(searches)
        return self.find_ents_from_matches(matches)


    def find_ents_db(self) -> List[Span]:
        matches = self.find_matches_db()
        return self.find_ents_from_matches(matches)
    

    def find_ents(self, *, verbose=False) -> List[Span]:
        ents_tn = self.find_ents_tn()

        if verbose:
            print('TN Entities:', ents_tn)
        
        ents_db = self.find_ents_db()

        if verbose:
            print('DB Entities:', ents_db)
        
        ents = filter_spans([*ents_tn, *ents_db])

        if not ents:
            return []
        
        # Merge Consecutive Span-Intervals
        ents.sort(key=lambda ent: ent.start)
        merged = ents and [ents[0]]
        for curr in ents[1:]:
            prev = merged[-1]
            if prev.end >= curr.start:
                end = max(prev.end, curr.end)
                merged[-1] = self.doc[prev.start:end]
            else:
                merged.append(curr)
        
        if verbose:
            print('Found Entities:', merged)
        
        return merged
    

    def find_substitutions(self) -> Dict[str, List[str]]:
        spans = [*self.doc.ents]
        spans.sort(key=lambda span: span.start)
        
        token_to_span = {}
        for span in spans:
            for token in span:
                token_to_span[token] = span
        
        # The text provides information about 
        # names that will be used interchangeably,
        # like "predatory crab (Carcinus maenas)".
        substitutions: Dict[str, List[str]] = {}


        def add_substitutions(a, b):
            notation_a = re_taxonomic_notation(a)
            notation_b = re_taxonomic_notation(b)

            a = a if notation_a else a.lower()
            b = b if notation_b else b.lower()
                    
            if a not in substitutions:
                substitutions[a] = []
            substitutions[a].append(b)

            return


        start = None
        for token in self.doc:
            # Neighboring Spans
            # We're only using the last token of the span.
            if token in token_to_span and token_to_span[token][-1] == token:
                token_span = token_to_span[token]
                token_n1 = token.i + 1 <= len(self.doc) - 1 and token.nbor(1)
                token_n2 = token.i + 2 <= len(self.doc) - 1 and token.nbor(2)
                
                same_sent_n2 = token and token_n2 and token.sent == token_n2.sent
                
                # Use Case: "Diptera: Tephritidae"
                if token_n1 and token_n2 and same_sent_n2 and token_n1.lower_ in ['.', ':'] and token_n2 in token_to_span:
                    add_substitutions(self.doc_text[token_span.start_char:token_span.end_char], self.doc_text[token_to_span[token_n2].start_char:token_to_span[token_n2].end_char])
            
            # Dealing with Parentheses
            # 1. Opening Parentheses
            if token.text == "(":
                start = token

            # 2. Closing Parentheses
            if start and token.text == ")":
                # Spans in Parentheses
                tracked_spans = set()
                for i in range(start.i + 1, token.i):
                    token = self.doc[i]
                    if token in token_to_span:
                        tracked_spans.add(token_to_span[token])

                # Span to Left of Parentheses
                sub_token = start.i != 0 and start.nbor(-1)
                if sub_token and sub_token in token_to_span:
                    sub_span = token_to_span[sub_token]
                    sub_span_text = self.doc_text[sub_span.start_char:sub_span.end_char]

                    for span in tracked_spans:
                        span_text = self.doc_text[span.start_char:span.end_char]
                        add_substitutions(sub_span_text, span_text)
                
                start = None
        
        
        return substitutions

In [21]:
class BaseEntityGroup(TypedDict):
    labels: Set[str]
    names: Set[str]
    lowers: Set[str]
    subs: Set[str]
    subs_mapped: Dict[str, Set[str]]
    forms: Set[str]
    taxa: Set[str]
    scientific: Set[str]
    vernacular: Set[str]
    taxonomic: Set[str]

EntityGroup = BaseEntityGroup | Dict[str, Set[str] | Dict[str, Set[str]]]

In [22]:
class MakeEntityGroups(Entity):
    def __init__(self, doc: Doc) -> None:
        super().__init__(doc)
    

    def create_entity_group(self, span: Span) -> EntityGroup:
        # Name
        name = self.doc_text[span.start_char:span.end_char]
        name_lower = name.lower()
        
        # Taxonomic Name
        name_taxonomic = name if re_taxonomic_name(name) else re_taxonomic_notation(name)
        name_taxonomic_lower = None if not name_taxonomic else name_taxonomic.lower()
        name_is_taxonomic_notation = name_taxonomic and name_taxonomic != name

        
        # Get Labels
        label = None
        
        if in_vernacular(name_lower):
            label = 'v'
        elif in_roles(name_lower):
            label = 'r'
        elif in_scientific(name_taxonomic_lower or name_lower):
            label = 's'
        else:
            for r_i, data in all_names.iter(self.doc_text_lower):
                if len(data['key']) <= 2:
                    continue

                l_i = r_i - len(data['key']) + 1
                r_i += 1

                l_bound = l_i == 0 or is_boundary(self.doc_text_lower[l_i-1])
                r_bound = r_i == len(self.doc_text_lower) or is_boundary(self.doc_text_lower[r_i])

                if not l_bound or not r_bound:
                    continue
                
                if label and label != data['label']:
                    label = 'm'
                    break
                label = data['label']
            

        # Get Substitutions
        subs = mapped_names.get(name_taxonomic_lower or name_lower, set())
        

        # Get Different Forms of Name
        def clean(s: str) -> str:
            s = re.sub(rf'[{string.punctuation}]', ' ', s)
            s = re.sub(r'\s+', ' ', s)
            return s
        
        # Get Forms
        forms = set()
        
        # Casing matters if the name is that
        # of taxonomic notation. For example,
        # Salmo trutta l. != Salmo trutta L.
        if name_is_taxonomic_notation:
            forms.add(name)
            forms.add(clean(name))
        else:
            forms.add(name_lower)
            forms.add(clean(name_lower))
        
        if name_taxonomic_lower:
            forms.add(name_taxonomic_lower)
        
        if label == 's':
            if name_taxonomic_lower:
                for inflection in abbreviate(name_taxonomic):
                    inflection = inflection.lower()
                    forms.add(inflection)
                    forms.add(clean(inflection))
            else:
                for inflection in get_inflections_latin(name_lower):
                    forms.add(inflection)
                    forms.add(clean(inflection))
        else:
            for inflection in get_inflections(name_lower, span[-1].tag_):
                forms.add(inflection)
                forms.add(clean(inflection))
        

        return {
            f'{label}': {name},
            'labels': set() if not label else {label},
            'names': {name},
            'forms': forms,
            'lowers': {name_lower},
            'subs': subs,
            'subs_mapped': {form: subs for form in forms},
            'taxa': forms if label == 's' and is_taxa(name) else set(),
            'scientific': forms if label == 's' else set(),
            # 'scientific_lowers': set() if label != 's' else {s.lower() for s in forms if in_scientific(s)},
            'vernacular': forms if label == 'v' else set(),
            'taxonomic': set() if not name_taxonomic_lower else {name_taxonomic_lower}
        }


    def merge_entity_group(self, a: EntityGroup, b: EntityGroup) -> EntityGroup:
        merged = {}
        
        keys = set([*a.keys(), *b.keys()])
        for key in keys:
            a_val = a.get(key, set())
            b_val = b.get(key, set())
            merged[key] = a_val | b_val # type: ignore
        
        return merged
    

    def merge_entity_groups(self, merge: Dict[int, Set[int]], groups: List[EntityGroup]) -> List[EntityGroup]:
        merge_items = list(merge.items())
        merge_items.sort(key=lambda item: len(item[1]), reverse=True)
        
        for source_i, targets_i in merge_items:
            if not groups[source_i]:
                continue
            
            for target_i in targets_i:
                if source_i == target_i:
                    continue
                
                if not groups[target_i]:
                    continue
                
                groups[target_i] = self.merge_entity_group(
                    groups[target_i], 
                    groups[source_i]
                )
            
            if [i for i in targets_i if groups[i]]:
                groups[source_i] = typing.cast(EntityGroup, None)

        return [group for group in groups if group]
    

    def group_by_contains(self, groups: List[EntityGroup]) -> List[EntityGroup]:
        merge = {i: set() for i in range(len(groups))}
        
        i = 0
        while i < len(groups):
            j = i + 1
            while j < len(groups):
                # Matching Labels
                # If the labels don't match, we can save time
                # and continue on to the next pair of groups.
                if (
                    'm' not in groups[i]['labels'] and 
                    'm' not in groups[j]['labels'] and
                    not ('r' in groups[i]['labels'] and 'v' in groups[j]['labels']) and
                    not ('v' in groups[i]['labels'] and 'r' in groups[j]['labels']) and
                    groups[i]['labels'].isdisjoint(groups[j]['labels']) # type: ignore
                ):
                    j += 1
                    continue
                
                forms_i: Any = groups[i]['forms']
                forms_j: Any = groups[j]['forms']

                if not forms_i.isdisjoint(forms_j):
                    merge[i].add(j)
                    merge[j].add(i)
                else:
                    found = [False, False]

                    for form_i in forms_i:
                        for form_j in forms_j:
                            index = form_j.find(form_i)
                            if index != -1 and (index == 0 or is_boundary(form_j[index-1])) and (index + len(form_i) == len(form_j) or is_boundary(form_j[index + len(form_i)])):
                                merge[i].add(j)
                                found[0] = True
                            index = form_i.find(form_j)
                            if index != -1 and (index == 0 or is_boundary(form_i[index-1])) and (index + len(form_j) == len(form_i) or is_boundary(form_i[index + len(form_j)])):
                                merge[j].add(i)
                                found[1] = True
                            
                            # No More Comparisons
                            # If both are true, we will not find
                            # any more important information. So,
                            # we can save ourselves the time.
                            if found[0] and found[1]:
                                break
                        
                        # Break
                        if found[0] and found[1]:
                            break
                
                j += 1
            i += 1

        return self.merge_entity_groups(merge, groups)


    def group_by_internal_subs(self, groups: List[EntityGroup], subs: Dict[str, List[str]]) -> List[EntityGroup]:
        merge = {i: set() for i in range(len(groups))}
        
        for k, v in subs.items():
            sources = [i for i, group in enumerate(groups) if group['forms'] & {k}] # type: ignore
            targets = [i for i, group in enumerate(groups) if i not in sources and group['forms'] & set(v)] # type: ignore
            
            # Swap
            # Either 'sources' or 'targets' will have only 1
            # element.
            if len(sources) > len(targets):
                sources, targets = targets, sources

            if not sources:
                continue
            
            merge[sources[0]].update(targets)
        
        return self.merge_entity_groups(merge, groups)


    def group_by_external_subs(self, groups: List[EntityGroup]) -> List[EntityGroup]:
        merge = {i: set() for i in range(len(groups))}
        
        i = 0
        while i < len(groups):
            j = i + 1
            while j < len(groups):
                if i in merge.get(j, []):
                    j += 1
                    continue

                found = False

                pairs = [
                    ['taxa', 'vernacular'],
                    ['vernacular', 'taxa'],
                    ['scientific', 'vernacular'],
                    ['vernacular', 'scientific']
                ]
                
                for l1, l2 in pairs:
                    A: Set[str] = groups[i][l1] - groups[j][l1] # type: ignore
                    B: Set[str] = groups[j][l2] - groups[i][l2] # type: ignore
                    
                    # We do not want to merge two different species because of a
                    # similar ancestor. Alas, the 'taxa' group cannot have any species.
                    if l1 == 'taxa' and groups[i]['taxonomic']:
                        continue

                    if l2 == 'taxa' and groups[j]['taxonomic']:
                        continue

                    subs_mapped_a: Dict[str, Set[str]] = groups[i]['subs_mapped'] # type: ignore
                    subs_mapped_b: Dict[str, Set[str]] = groups[j]['subs_mapped'] # type: ignore

                    A_subs = set().union(*[subs_mapped_a[a] for a in A if subs_mapped_a[a]]) or A
                    B_subs = set().union(*[subs_mapped_b[b] for b in B if subs_mapped_b[b]]) or B
                    
                    if not A_subs.isdisjoint(B_subs):
                        merge[i].add(j)
                        merge[j].add(i)
                        found = True
                        break
                    
                if not found:
                    A: Set[str] = groups[i]['forms'] - groups[j]['forms'] # type: ignore
                    B: Set[str] = groups[j]['forms'] - groups[i]['forms'] # type: ignore

                    for a in A:
                        for b in B:
                            if names_related(a, b):
                                merge[i].add(j)
                                merge[j].add(i)
                                found = True
                                break
                            
                        if found:
                            break
                j += 1
            i += 1

        return self.merge_entity_groups(merge, groups)


    def group_ents(self, ents: List[Span], *, verbose=False) -> List[EntityGroup]:
        mapped_ents = {}
        for ent in ents:
            mapped_ents[self.doc_text_lower[ent.start_char:ent.end_char]] = ent
        
        groups: Any = [self.create_entity_group(ent) for ent in mapped_ents.values()]

        if verbose:
            print('Create Groups')
            for g_i, group in enumerate(groups):
                print(g_i, group['forms'], group['labels'])
            print()
            print()
        
        groups = self.group_by_contains(groups)    
        if verbose:
            print('Group by Contains')
            for g_i, group in enumerate(groups):
                print(g_i, group['forms'])
            print()
            print()
        
        subs = self.find_substitutions()
        if verbose:
            print(f'Subs: {subs}')
        
        groups = self.group_by_internal_subs(groups, subs)
        if verbose:
            print('Group by Internal Subs')
            for g_i, group in enumerate(groups):
                print(g_i, group['forms'])
            print()
            print()

        groups = self.group_by_external_subs(groups)
        if verbose:
            print('Group by External Subs')
            for g_i, group in enumerate(groups):
                print(g_i, group['forms'])
            print()
            print()

        if verbose:
            print()
            for g_i, group in enumerate(groups):
                print(g_i, group['forms'])
            print()
            print()

        return groups

In [23]:
df = pd.read_excel("./benchmarks/Benchmark-03-09.xlsx")
df = df[(df['Score'] == 0) | (df['Score'] == 1) | (df['Score'] == 3)]
df.reset_index(inplace=True)
texts = df.Abstract.tolist()
number_texts = len(texts)

In [24]:
df_T = df[df['Score'] == 3]
texts_T = df_T.Abstract.tolist()
number_texts_T = len(texts_T)

In [25]:
df_F = df[(df['Score'] < 3) & (df['Score'] >= 0)]
texts_F = df_F.Abstract.tolist()
number_texts_F = len(texts_F)

In [26]:
# type: ignore
nlp = TaxoNERD(prefer_gpu=True).load(
    model="en_ner_eco_biobert", 
    exclude=["tok2vec", "parser", "lemmatizer"]
)
spacy.require_gpu()

True

In [27]:
# nlp = TaxoNERD(prefer_gpu=False).load(
#     model="en_ner_eco_biobert", 
#     exclude=["tok2vec", "parser", "lemmatizer"]
# )

In [28]:
def process_string(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = regex.sub(r'(\p{Ll})(\p{Lu})', r'\1 \2', s)
    s = regex.sub(r'([\.,:;\)\]\}])([\p{Lu}\p{Ll}])', r'\1 \2', s)
    s = regex.sub(r'([\p{Lu}\p{Ll}])([\(\[\{]])', r'\1 \2', s)
    return s

In [29]:
def evaluate_groups(mkg: MakeEntityGroups, ents: List[Span], groups: List[Any], *, verbose=False) -> bool:
    if verbose:
        print('Ents')
        for ent in ents:
            print(ent)
        print()
        print()

    # Store Counts
    counts = {}
    for ent in ents:
        ent_text = mkg.doc_text_lower[ent.start_char:ent.end_char]
        counts[ent_text] = counts.get(ent_text, 0) + 1
    total_count = sum(counts.values())
    
    if verbose:
        print('Counts')
        for k, v in counts.items():
            print(v, k)
        print()
        print()
    
    if verbose:
        print('Group\'s Lowers')
        for group in groups:
            print(group['lowers'])
        print()
        print()

    lowers: Any = {}
    for group in groups:
        for lower in group['lowers']:
            lowers[lower] = lowers.get(lower, 0) +  1

    if verbose:
        print('Lowers')
        for k, v in lowers.items():
            print(v, k)
        print()
        print()

    for group in groups:
        count = sum([counts[lower] for lower in group['lowers']])
        group['count'] = count
        if verbose:
            print(group['lowers'], group['count'], group['count'] / total_count)

    flag = len([group for group in groups if {'v', 'r', 'm'} & group['labels']]) >= 2 and (len([group for group in groups if group['count'] >= 2 and {'v', 'r', 'm'} & group['labels']]) >= 2 or len([group for group in groups if group['count'] / total_count >= 0.5]) >= 1)
    if verbose:
        print('Flagged', flag)
        print()
        print()
    
    return flag

In [30]:
# # type: ignore
# import cProfile, pstats, io
# import matplotlib.pyplot as plt
# from sklearn.metrics import ConfusionMatrixDisplay

# # Clear Cache
# import gc
# import functools

# gc.collect()
# wrappers = [
#     a for a in gc.get_objects() 
#     if isinstance(a, functools._lru_cache_wrapper)]

# for wrapper in wrappers:
#     wrapper.cache_clear()

# txts = [process_string(text) for text in texts]
# n_txts = len(txts)

# y_pred = []
# y_true = []
# bad = []

# verbose = False
# # pr = cProfile.Profile()
# # pr.enable()

# i = 0
# for doc in tqdm(nlp.pipe(txts, batch_size=1), total=n_txts):
#     # if i not in [0, 6, 16, 26, 33]:
#     #     i += 1
#     #     continue
#     # else:
#     #     verbose = True
#     #     print()
#     #     print('Text', i)
#     #     print(doc.text)
#     #     print()
#     #     print()
    
#     mkg = MakeEntityGroups(doc)
#     ents = mkg.find_ents(verbose=verbose)
#     ent_groups: List[Any] = mkg.group_ents(ents, verbose=verbose)

#     flag = evaluate_groups(mkg, ents, ent_groups)

#     y_pred.append(flag)
#     y_true.append(df.loc[i, 'Score'] == 3)

#     if y_true[-1] != y_pred[-1]:
#         bad.append(i)

#     i += 1

# ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
# plt.show()

# # pr.disable()
# # ps = pstats.Stats(pr)
# # ps.sort_stats('cumulative')
# # ps.print_callees('find_ents') 

# print(bad)

In [31]:
filename = "4-ScreenByEntities"


def save(*, mask, counts, errors, suffix):
    outputs = {
        "counts": counts,
        "mask": mask,
        "errors": errors
    }
    
    with open(rf'{filename}-{suffix or 0}.pickle', 'wb') as file:
        pickle.dump(outputs, file)


def load(suffix):
    outputs = {
        "errors": {},
        "counts": {
            None: 0, 
            True: 0, 
            False: 0
        },
        "mask": []
    }

    fn = f'{filename}-{suffix or 0}.pickle'

    # Nothing To Load
    if not os.path.isfile(fn):
        return outputs

    with open(fn, 'rb') as file:
        outputs = pickle.load(file)
    
    return outputs

In [32]:
# type: ignore
suffix = None

outputs = load(suffix)
counts = outputs['counts']
errors = outputs['errors']
mask = outputs['mask']

papers = pd.read_csv('3_Papers_Screened.csv')
texts = papers.Abstract.tolist()
number_texts = len(texts)

i = len(mask)
progress = tqdm(nlp.pipe(texts, batch_size=128), total=number_texts)
for doc in progress:
    progress.set_postfix({'Errors': counts[None], 'Included': counts[True], 'Excluded': counts[False]}, refresh=True)

    flag = None
    
    # Auto-Save
    if (i + 1) % 256:
        save(mask=mask, counts=counts, errors=errors, suffix=suffix)
    
    try:
        make_ent_groups = MakeEntityGroups(doc)
        ents = make_ent_groups.find_ents()
        ent_groups: List[Any] = make_ent_groups.group_ents(ents)
        flag = evaluate_groups(make_ent_groups, ents, ent_groups)
    except Exception as e:
        errors[i] = e

    counts[flag] += 1
    mask.append(flag)
    
    i += 1

100%|██████████| 27783/27783 [39:44<00:00, 11.65it/s, Errors=0, Included=24197, Excluded=3585]


In [34]:
errors

{}